In [2]:
# se importan librerias
import pandas as pd
import numpy as np

In [3]:
# se carga el dataset
df = pd.read_csv('../data/customer_acquisition_data.csv')
df.head()

,customer_id,channel,cost,conversion_rate,revenue
0,1,referral,8.320327,0.123145,4199
1,2,paid advertising,30.450327,0.016341,3410
2,3,email marketing,5.246263,0.043822,3164
3,4,social media,9.546326,0.167592,1520
4,5,referral,8.320327,0.123145,2419


In [4]:
# se crea la columna net_value
df['net_value'] = df['revenue'] - df['cost'].round(3)
df.head()

,customer_id,channel,cost,conversion_rate,revenue,net_value
0,1,referral,8.320327,0.123145,4199,4190.680
1,2,paid advertising,30.450327,0.016341,3410,3379.550
2,3,email marketing,5.246263,0.043822,3164,3158.754
3,4,social media,9.546326,0.167592,1520,1510.454
4,5,referral,8.320327,0.123145,2419,2410.680


In [5]:
df[['revenue','net_value','conversion_rate','cost']].describe()

,revenue,net_value,conversion_rate,cost
count,800.000000,800.000000,800.000000,800.000000
mean,2769.151250,2756.003507,0.086305,13.148052
std,1259.543706,1259.397811,0.059611,9.922337
min,500.000000,479.550000,0.016341,5.246263
25%,1694.000000,1681.101000,0.043822,5.246263
50%,2764.000000,2754.104000,0.043822,8.320327
75%,3824.250000,3809.228000,0.123145,9.546326
max,4998.000000,4985.680000,0.167592,30.450327


## Segmentacion de Clientes
- definir variables para segmentación;
- determinar criterios High/Medium/Low Value
- crear segmentos
- analizar los clientes de cada segmento;
- identificar los clientes más valiosos.

### Definimos variables y criterios de segmentacion
high_value : Revenue > P75
Net Value > P75
Conversion > P75
Cost <= P25

medium_value : P50 < Revenue <= P75
P50 < Net Value <= P75
P50 < Conversion <= P75
P25 < Cost <= P50

low_value : P25 < Revenue <= P50
P25 < Net Value <= P50
P25 < Conversion <= P50
P50 < Cost <= P75

Other: Diferente a los criterios establecidos

In [6]:
# definimos la funcion para segmentar los clientes

def segment_customer(df):

    # Revenue percentiles
    revenue_p25 = df["revenue"].quantile(0.25)
    revenue_p50 = df["revenue"].quantile(0.50)
    revenue_p75 = df["revenue"].quantile(0.75)

    # Net value percentiles
    net_value_p25 = df["net_value"].quantile(0.25)
    net_value_p50 = df["net_value"].quantile(0.50)
    net_value_p75 = df["net_value"].quantile(0.75)

    # Conversion rate percentiles
    conversion_p25 = df["conversion_rate"].quantile(0.25)
    conversion_p50 = df["conversion_rate"].quantile(0.50)
    conversion_p75 = df["conversion_rate"].quantile(0.75)

    # Cost percentiles
    cost_p25 = df["cost"].quantile(0.25)
    cost_p50 = df["cost"].quantile(0.50)
    cost_p75 = df["cost"].quantile(0.75)

    # High Value
    high_value = (
        (df["revenue"] > revenue_p75) &
        (df["net_value"] > net_value_p75) &
        (df["conversion_rate"] > conversion_p75) &
        (df["cost"] <= cost_p25)
    )

    # Medium Value
    medium_value = (
        (df["revenue"] > revenue_p50) &
        (df["revenue"] <= revenue_p75) &
        (df["net_value"] > net_value_p50) &
        (df["net_value"] <= net_value_p75) &
        (df["conversion_rate"] > conversion_p50) &
        (df["conversion_rate"] <= conversion_p75) &
        (df["cost"] > cost_p25) &
        (df["cost"] <= cost_p50)
    )

    # Low Value
    low_value = (
        (df["revenue"] > revenue_p25) &
        (df["revenue"] <= revenue_p50) &
        (df["net_value"] > net_value_p25) &
        (df["net_value"] <= net_value_p50) &
        (df["conversion_rate"] > conversion_p25) &
        (df["conversion_rate"] <= conversion_p50) &
        (df["cost"] > cost_p50) &
        (df["cost"] <= cost_p75)
    )

    # Assign segments
    df["customer_segment"] = np.select(
        [high_value, medium_value, low_value],
        ["High Value", "Medium Value", "Low Value"],
        default="Other"
    )

    return df

In [7]:
# aplicamos la funcion para segmentar los clientes
df = segment_customer(df)
df.head()

,customer_id,channel,cost,conversion_rate,revenue,net_value,customer_segment
0,1,referral,8.320327,0.123145,4199,4190.680,Other
1,2,paid advertising,30.450327,0.016341,3410,3379.550,Other
2,3,email marketing,5.246263,0.043822,3164,3158.754,Other
3,4,social media,9.546326,0.167592,1520,1510.454,Other
4,5,referral,8.320327,0.123145,2419,2410.680,Other


In [8]:
# revisamos porcentaje de clientes hay en cada segmento
df["customer_segment"].value_counts(normalize=True) * 100

customer_segment
Other           93.125
Medium Value     6.875
Name: proportion, dtype: float64

**Metodologia de segmentacion Inicial** 

Los resultados muestran que la metodología utilizada para clasificar a los clientes es demasiado restrictiva. El 93,12% de los clientes fue clasificado como Other, lo que indica que la mayoría no cumple simultáneamente con los criterios establecidos para los segmentos High, Medium o Low Value.

Este resultado evidencia la necesidad de ajustar la metodología de segmentación y desarrollar un enfoque más flexible que permita identificar de manera efectiva a los clientes con mayor valor comercial para la empresa. 

In [9]:
# utilizamos la metodologia customer_value_score
def customer_value_score(df):

    # Calculate thresholds

    revenue_p75 = df["revenue"].quantile(0.75)

    net_value_p75 = df["net_value"].quantile(0.75)

    conversion_rate_p75 = df["conversion_rate"].quantile(0.75)

    cost_p25 = df["cost"].quantile(0.25)

    # Initialize score

    df["customer_value_score"] = 0

    # Revenue: +2 points

    df.loc[
        df["revenue"] > revenue_p75,
        "customer_value_score"
    ] += 2

    # Net value: +2 points

    df.loc[
        df["net_value"] > net_value_p75,
        "customer_value_score"
    ] += 2

    # Conversion rate: +1 point

    df.loc[
        df["conversion_rate"] > conversion_rate_p75,
        "customer_value_score"
    ] += 1

    # Cost: +1 point

    df.loc[
        df["cost"] <= cost_p25,
        "customer_value_score"
    ] += 1

    return df

In [10]:
df = customer_value_score(df)
df["customer_value_score"].value_counts(normalize=True).sort_index() * 100

customer_value_score
0    37.125
1    37.875
4    13.000
5    12.000
Name: proportion, dtype: float64

In [11]:
df["customer_value_score"].value_counts()

customer_value_score
1    303
0    297
4    104
5     96
Name: count, dtype: int64

In [12]:
# metricas de customer_value_score
df.groupby("customer_value_score")[
    ["revenue", "net_value", "conversion_rate", "cost"]
].mean()

,revenue,net_value,conversion_rate,cost
customer_value_score,,,,
0,2229.555556,2211.027441,0.073879,18.528441
1,2231.313531,2224.123307,0.099784,7.190516
4,4382.403846,4361.954904,0.064608,20.449269
5,4388.385417,4380.989417,0.105707,7.396294


In [13]:
df["conversion_rate"].describe()

count    800.000000
mean       0.086305
std        0.059611
min        0.016341
25%        0.043822
50%        0.043822
75%        0.123145
max        0.167592
Name: conversion_rate, dtype: float64

In [14]:
df["conversion_rate"].value_counts().sort_index()

conversion_rate
0.016341    194
0.043822    214
0.123145    207
0.167592    185
Name: count, dtype: int64

In [15]:
df["conversion_rate"].nunique()

4

In [16]:
df.groupby("channel")["conversion_rate"].mean().sort_values(ascending=False)*100

channel
social media        16.759225
referral            12.314498
email marketing      4.382223
paid advertising     1.634149
Name: conversion_rate, dtype: float64

**Metodología inicial de Customer Value Score**

Inicialmente, se utilizó una metodología de Customer Value Score, que consiste en asignar una puntuación a cada cliente de acuerdo con su desempeño en las principales variables analizadas:

Revenue > P75 → 2 puntos
Net Value > P75 → 2 puntos
Conversion Rate > P75 → 1 punto
Cost ≤ P25 → 1 punto

Donde P representa el percentil correspondiente dentro de la muestra.

Sin embargo, al analizar los resultados, identificamos que la variable conversion_rate podría ser demasiado restrictiva para determinar el valor de un cliente. Encontramos, por ejemplo, que el canal Paid Advertising presenta una tasa de conversión relativamente baja, pero al mismo tiempo registra uno de los niveles más altos de revenue y net_value.

Esto sugiere que una tasa de conversión elevada no necesariamente representa un mayor valor económico por cliente. Además, se identificó que conversion_rate está directamente relacionada con el canal de adquisición y presenta únicamente cuatro valores dentro del dataset.

Por esta razón, se decidió excluir conversion_rate del cálculo principal del Customer Value Score y utilizarla posteriormente como una métrica complementaria para analizar el desempeño de los canales y desarrollar estrategias de marketing más específicas.

La metodología de segmentación se centrará principalmente en las variables que representan directamente el valor económico generado por cada cliente: revenue, net_value y cost.


In [17]:
# modificamos la evaluacio de customer_value_score solo con las variables revenue, net_value y cost
def customer_value_score_net(df):

    # Calculate thresholds

    revenue_p75 = df["revenue"].quantile(0.75)

    net_value_p75 = df["net_value"].quantile(0.75)

    cost_p25 = df["cost"].quantile(0.25)

    # Initialize score

    df["customer_value_score_net"] = 0

    # Revenue: +2 points

    df.loc[
        df["revenue"] > revenue_p75,
        "customer_value_score_net"
    ] += 2

    # Net value: +2 points

    df.loc[
        df["net_value"] > net_value_p75,
        "customer_value_score_net"
    ] += 2

    # Cost: +1 point

    df.loc[
        df["cost"] <= cost_p25,
        "customer_value_score_net"
    ] += 1

    return df

In [18]:
df = customer_value_score_net(df)
df["customer_value_score_net"].value_counts(normalize=True).sort_index() * 100


customer_value_score_net
0    54.25
1    20.75
4    19.00
5     6.00
Name: proportion, dtype: float64

In [19]:
df["customer_value_score_net"].value_counts().sort_index()

customer_value_score_net
0    434
1    166
4    152
5     48
Name: count, dtype: int64

In [20]:
df.groupby("customer_value_score_net")[
    ["revenue", "net_value", "cost"]
].mean()

,revenue,net_value,cost
customer_value_score_net,,,
0,2182.543779,2166.851032,15.693073
1,2355.674699,2350.428699,5.246263
4,4364.414474,4347.408566,17.006234
5,4451.333333,4446.087333,5.246263


**Metodología de segmentación basada en valor económico**

Se estableció una metodología de segmentación basada en tres métricas y sus respectivos umbrales:

* **Revenue > P75:** +2 puntos
* **Net Value > P75:** +2 puntos
* **Cost ≤ P25:** +1 punto

A partir del puntaje obtenido, y bajo un criterio de negocio, se definieron los siguientes segmentos:

* **High Value — High Efficiency:** 5 puntos
* **High Value — Standard Efficiency:** 4 puntos
* **Cost Efficient / Potential Value:** 1 punto
* **Low Value:** 0 puntos

Los resultados muestran que el **6% de los clientes** pertenece al segmento **High Value — High Efficiency**. Estos clientes presentan simultáneamente un alto revenue, un alto net value y un costo relativamente bajo, por lo que representan el grupo con mayor valor económico y eficiencia dentro de la base analizada.

El **19% de los clientes** pertenece al segmento **High Value — Standard Efficiency**. Aunque estos clientes presentan niveles elevados de revenue y net value, su costo promedio es superior al del segmento de High Value — High Efficiency. Esto representa una oportunidad para realizar un análisis adicional por canal y determinar qué factores están asociados con el mayor costo de estos clientes.

Por otro lado, el **54% de los clientes** pertenece al segmento **Low Value**. Este grupo presenta un revenue promedio de aproximadamente **USD 2.183** y un net value promedio de **USD 2.167**, con un costo promedio cercano a **USD 15,69**. Debido al tamaño de este segmento, representa una importante oportunidad de mejora para desarrollar estrategias orientadas a incrementar el revenue y el net value de estos clientes, manteniendo controlados los costos.

Finalmente, el **21% restante** corresponde al segmento **Cost Efficient / Potential Value**, caracterizado por presentar un costo relativamente bajo, pero sin alcanzar los umbrales de revenue y net value establecidos para los segmentos de alto valor. Este grupo podría representar una oportunidad para desarrollar estrategias de crecimiento y aumentar su valor futuro.


In [21]:
# converitimos el customer_value_score en una categoria de negocio. 
def assign_customer_segment(df):

    df["customer_segment"] = np.select(
        [
            df["customer_value_score_net"] == 5,
            df["customer_value_score_net"] == 4,
            df["customer_value_score_net"] == 1,
            df["customer_value_score_net"] == 0
        ],
        [
            "High Value - High Efficiency",
            "High Value - Standard Efficiency",
            "Cost Efficient - Potential Value",
            "Low Value"
        ],
        default="Other"
    )

    return df

In [22]:
# revisamos la cantidad de clientes en cada segmento
df = assign_customer_segment(df)
df["customer_segment"].value_counts()

customer_segment
Low Value                           434
Cost Efficient - Potential Value    166
High Value - Standard Efficiency    152
High Value - High Efficiency         48
Name: count, dtype: int64

In [23]:
segment_channel = pd.crosstab(
    df["customer_segment"],
    df["channel"],
    normalize="index"
) * 100

segment_channel

channel,email marketing,paid advertising,referral,social media
customer_segment,,,,
Cost Efficient - Potential Value,100.0,0.00000,0.000000,0.000000
High Value - High Efficiency,100.0,0.00000,0.000000,0.000000
High Value - Standard Efficiency,0.0,37.50000,30.921053,31.578947
Low Value,0.0,31.56682,36.866359,31.566820


In [24]:
# valor economico por segmento y canal
df.groupby(
    ["customer_segment", "channel"]
)[["revenue", "net_value", "cost", "conversion_rate"]].mean()

revenue    net_value  \
customer_segment                 channel                                      
Cost Efficient - Potential Value email marketing   2355.674699  2350.428699   
High Value - High Efficiency     email marketing   4451.333333  4446.087333   
High Value - Standard Efficiency paid advertising  4392.912281  4362.462281   
                                 referral          4369.659574  4361.339574   
                                 social media      4325.437500  4315.891500   
Low Value                        paid advertising  2175.182482  2144.732482   
                                 referral          2276.112500  2267.792500   
                                 social media      2080.627737  2071.081737   

                                                        cost  conversion_rate  
customer_segment                 channel                                       
Cost Efficient - Potential Value email marketing    5.246263         0.043822  
High Value - High Efficiency     email marketing    5.246263         0.043822  
High Value - Standard Efficiency paid advertising  30.450327         0.016341  
                                 referral           8.320327         0.123145  
                                 social media       9.546326         0.167592  
Low Value                        paid advertising  30.450327         0.016341  
                                 referral           8.320327         0.123145  
                                 social media       9.546326         0.167592

### Análisis de segmentos por canal

El análisis de la distribución de los segmentos por canal permite identificar diferencias importantes entre eficiencia de adquisición, conversión y valor económico generado por cliente.

**Email Marketing** concentra el **100% de los clientes clasificados como High Value — High Efficiency**. Este segmento representa el 6% de la base total y presenta un revenue promedio de USD 4.451 y un net value promedio de USD 4.446, con un costo promedio de apenas USD 5,25. Esto evidencia que Email Marketing es un canal especialmente eficiente para generar clientes de alto valor.

**Paid Advertising** genera clientes pertenecientes al segmento High Value — Standard Efficiency y presenta un revenue promedio de aproximadamente USD 4.393. Sin embargo, registra un costo promedio de USD 30,45, considerablemente superior al resto de los canales. Esto representa una oportunidad para analizar cómo reducir el costo de adquisición manteniendo el alto valor económico de los clientes obtenidos.

**Social Media** presenta la mayor tasa de conversión, con 16,76%; sin embargo, una mayor conversión no se traduce necesariamente en un mayor valor económico por cliente. Esto evidencia que la tasa de conversión debe analizarse conjuntamente con revenue y net value para evaluar el desempeño real de un canal.

Finalmente, **Email Marketing presenta una oportunidad adicional de crecimiento** dentro del segmento Cost Efficient — Potential Value. Los 166 clientes de este segmento presentan un revenue promedio de USD 2.356 y un net value de USD 2.350, manteniendo el mismo costo promedio de USD 5,25 observado en el segmento High Value — High Efficiency. Por lo tanto, existe una oportunidad para desarrollar estrategias de retención, cross-selling y up-selling orientadas a incrementar el valor de estos clientes sin aumentar significativamente el costo de adquisición.

En conjunto, los resultados muestran que **el canal con mayor conversión no necesariamente es el canal que genera mayor valor económico**, mientras que canales con menor conversión pueden generar clientes de mayor valor. Por esta razón, las estrategias de marketing deben considerar conjuntamente el valor económico del cliente, el costo y el canal de adquisición.



In [26]:
# se exporta el dataset con las nuevas columnas, para exportar a la siguiente rama
df.to_csv('../data/customer_acquisition_data_segmented.csv', index=False)
df.head()

,customer_id,channel,cost,conversion_rate,revenue,net_value,customer_segment,customer_value_score,customer_value_score_net
0,1,referral,8.320327,0.123145,4199,4190.680,High Value - Standard Efficiency,4,4
1,2,paid advertising,30.450327,0.016341,3410,3379.550,Low Value,0,0
2,3,email marketing,5.246263,0.043822,3164,3158.754,Cost Efficient - Potential Value,1,1
3,4,social media,9.546326,0.167592,1520,1510.454,Low Value,1,0
4,5,referral,8.320327,0.123145,2419,2410.680,Low Value,0,0
